In [1]:
import pandas as pd
import os

# ============================================================
# FILE PATHS
# ============================================================

files = {
    "OUTPATIENT": "../data/processed/primary/outpatient_claims_with_beneficiary_features.csv",
    "INPATIENT": "../data/processed/primary/inpatient_claims_fully_enriched.csv",
    "CARRIER": "../data/processed/primary/carrier_claims_with_beneficiary_features.csv",
}

# ============================================================
# LOAD + BASIC INFORMATION
# ============================================================

dfs = {}

for name, path in files.items():

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    df = pd.read_csv(path, low_memory=False)
    dfs[name] = df

    print("Shape:", df.shape)
    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    print("\nColumns:")
    for col in df.columns:
        print(" ", col)

# ============================================================
# FIND COMMON COLUMNS
# ============================================================

column_sets = {
    name: set(df.columns)
    for name, df in dfs.items()
}

common_all = set.intersection(*column_sets.values())

print("\n" + "=" * 80)
print("COLUMNS COMMON TO ALL THREE CLAIM DATASETS")
print("=" * 80)

for col in sorted(common_all):
    print(col)

# ============================================================
# COMMON BETWEEN PAIRS
# ============================================================

print("\n" + "=" * 80)
print("OUTPATIENT ∩ INPATIENT")
print("=" * 80)

for col in sorted(column_sets["OUTPATIENT"] & column_sets["INPATIENT"]):
    print(col)

print("\n" + "=" * 80)
print("OUTPATIENT ∩ CARRIER")
print("=" * 80)

for col in sorted(column_sets["OUTPATIENT"] & column_sets["CARRIER"]):
    print(col)

print("\n" + "=" * 80)
print("INPATIENT ∩ CARRIER")
print("=" * 80)

for col in sorted(column_sets["INPATIENT"] & column_sets["CARRIER"]):
    print(col)

# ============================================================
# UNIQUE COLUMNS BY DATASET
# ============================================================

for name in dfs:

    other_columns = set.union(
        *(column_sets[x] for x in dfs if x != name)
    )

    unique_columns = column_sets[name] - other_columns

    print("\n" + "=" * 80)
    print(f"UNIQUE COLUMNS: {name}")
    print("=" * 80)

    for col in sorted(unique_columns):
        print(col)

# ============================================================
# POSSIBLE IDENTIFIER COLUMNS
# ============================================================

print("\n" + "=" * 80)
print("IDENTIFIER / KEY CANDIDATES")
print("=" * 80)

identifier_keywords = [
    "ID",
    "KEY",
    "NPI",
    "PRVDR",
    "PROVIDER"
]

for name, df in dfs.items():

    print(f"\n--- {name} ---")

    for col in df.columns:

        upper = col.upper()

        if any(keyword in upper for keyword in identifier_keywords):
            print(col)

# ============================================================
# NUMERIC / CATEGORICAL / DATE-LIKE COLUMNS
# ============================================================

for name, df in dfs.items():

    print("\n" + "=" * 80)
    print(f"DATA TYPES: {name}")
    print("=" * 80)

    print("\nNumeric columns:")
    for col in df.select_dtypes(include="number").columns:
        print(" ", col)

    print("\nObject/category columns:")
    for col in df.select_dtypes(include=["object", "category"]).columns:
        print(" ", col)

# ============================================================
# MISSINGNESS FOR EACH DATASET
# ============================================================

for name, df in dfs.items():

    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    print("\n" + "=" * 80)
    print(f"MISSING VALUES: {name}")
    print("=" * 80)

    if len(missing) == 0:
        print("No missing values.")
    else:
        print(missing)

# ============================================================
# BENEFICIARY FEATURE CHECK
# ============================================================

beneficiary_features = [
    "BENE_BIRTH_DT",
    "BENE_DEATH_DT",
    "BENE_SEX_IDENT_CD",
    "BENE_RACE_CD",
    "BENE_ESRD_IND",
    "SP_STATE_CODE",
    "BENE_COUNTY_CD",
    "BENE_HI_CVRAGE_TOT_MONS",
    "BENE_SMI_CVRAGE_TOT_MONS",
    "BENE_HMO_CVRAGE_TOT_MONS",
    "PLAN_CVRG_MOS_NUM",
    "SP_ALZHDMTA",
    "SP_CHF",
    "SP_CHRNKIDN",
    "SP_CNCR",
    "SP_COPD",
    "SP_DEPRESSN",
    "SP_DIABETES",
    "SP_ISCHMCHT",
    "SP_OSTEOPRS",
    "SP_RA_OA",
    "SP_STRKETIA",
    "MEDREIMB_IP",
    "BENRES_IP",
    "PPPYMT_IP",
    "MEDREIMB_OP",
    "BENRES_OP",
    "PPPYMT_OP",
    "MEDREIMB_CAR",
    "BENRES_CAR",
    "PPPYMT_CAR",
    "YEAR"
]

print("\n" + "=" * 80)
print("BENEFICIARY FEATURE AVAILABILITY")
print("=" * 80)

for name, df in dfs.items():

    available = [
        col for col in beneficiary_features
        if col in df.columns
    ]

    missing = [
        col for col in beneficiary_features
        if col not in df.columns
    ]

    print(f"\n{name}")
    print("Beneficiary features available:", len(available))
    print("Beneficiary features missing:", len(missing))

    if missing:
        print("Missing:")
        for col in missing:
            print(" ", col)

# ============================================================
# CLAIM TYPE KEY CHECK
# ============================================================

print("\n" + "=" * 80)
print("CLAIM KEY / ID UNIQUENESS")
print("=" * 80)

for name, df in dfs.items():

    print(f"\n{name}")

    for col in ["CLAIM_KEY", "CLM_ID"]:

        if col in df.columns:

            print(
                f"{col}: "
                f"unique={df[col].nunique(dropna=False)}, "
                f"duplicates={df[col].duplicated().sum()}"
            )


OUTPATIENT
Shape: (790790, 57)
Rows: 790790
Columns: 57

Columns:
  CLAIM_KEY
  DESYNPUF_ID
  CLM_ID
  SEGMENT
  PRVDR_NUM
  CLM_PMT_AMT
  NCH_PRMRY_PYR_CLM_PD_AMT
  NCH_BENE_BLOOD_DDCTBL_LBLTY_AM
  NCH_BENE_PTB_DDCTBL_AMT
  NCH_BENE_PTB_COINSRNC_AMT
  TOTAL_REIMBURSEMENT
  CLM_FROM_DT
  CLM_THRU_DT
  CLAIM_DURATION_DAYS
  CLAIM_YEAR
  CLAIM_MONTH
  DIAGNOSIS_COUNT
  PROCEDURE_COUNT
  HCPCS_COUNT
  HAS_DIAGNOSIS
  HAS_PROCEDURE
  HAS_HCPCS
  HAS_NEGATIVE_PAYMENT
  HAS_PRIMARY_PAYER_PAYMENT
  IS_SEGMENT_2
  HAS_SEGMENT_1_MATCH
  BENE_BIRTH_DT
  BENE_DEATH_DT
  BENE_SEX_IDENT_CD
  BENE_RACE_CD
  BENE_ESRD_IND
  SP_STATE_CODE
  BENE_COUNTY_CD
  BENE_HI_CVRAGE_TOT_MONS
  BENE_SMI_CVRAGE_TOT_MONS
  BENE_HMO_CVRAGE_TOT_MONS
  PLAN_CVRG_MOS_NUM
  SP_ALZHDMTA
  SP_CHF
  SP_CHRNKIDN
  SP_CNCR
  SP_COPD
  SP_DEPRESSN
  SP_DIABETES
  SP_ISCHMCHT
  SP_OSTEOPRS
  SP_RA_OA
  SP_STRKETIA
  MEDREIMB_IP
  BENRES_IP
  PPPYMT_IP
  MEDREIMB_OP
  BENRES_OP
  PPPYMT_OP
  MEDREIMB_CAR
  BENRES_CAR
  PPPYMT_

In [2]:
import pandas as pd

# ============================================================
# LOAD THE THREE DATASETS
# ============================================================

outpatient = pd.read_csv(
    "../data/processed/primary/outpatient_claims_with_beneficiary_features.csv",
    low_memory=False
)

inpatient = pd.read_csv(
    "../data/processed/primary/inpatient_claims_fully_enriched.csv",
    low_memory=False
)

carrier = pd.read_csv(
    "../data/processed/primary/carrier_claims_with_beneficiary_features.csv",
    low_memory=False
)

# ============================================================
# CHECK CODE / ID COLUMNS
# ============================================================

datasets = {
    "OUTPATIENT": outpatient,
    "INPATIENT": inpatient,
    "CARRIER": carrier
}

for name, df in datasets.items():

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    cols_to_check = [
        "CLM_ID",
        "CLAIM_KEY",
        "DESYNPUF_ID",
        "PRVDR_NUM",
        "AT_PHYSN_NPI",
        "OP_PHYSN_NPI",
        "OT_PHYSN_NPI",
        "primary_provider_npi",
        "CLM_DRG_CD",
        "ADMTNG_ICD9_DGNS_CD",
        "ICD9_DGNS_CD_1",
        "ICD9_PRCDR_CD_1"
    ]

    for col in cols_to_check:

        if col in df.columns:

            print(f"\n{col}")
            print("dtype:", df[col].dtype)
            print("non-null:", df[col].notna().sum())
            print("unique:", df[col].nunique(dropna=True))

            sample = (
                df[col]
                .dropna()
                .astype(str)
                .head(5)
                .tolist()
            )

            print("sample:", sample)


OUTPATIENT

CLM_ID
dtype: int64
non-null: 790790
unique: 779815
sample: ['542192281063886', '542272281166593', '542282281644416', '542642281250669', '542242281386963']

CLAIM_KEY
dtype: object
non-null: 790790
unique: 790790
sample: ['542192281063886_1', '542272281166593_1', '542282281644416_1', '542642281250669_1', '542242281386963_1']

DESYNPUF_ID
dtype: object
non-null: 790790
unique: 85272
sample: ['00013D2EFD8E45D1', '00016F745862898F', '00016F745862898F', '0001FDD721E223DC', '00024B3D2352D2D0']

PRVDR_NUM
dtype: object
non-null: 790790
unique: 6294
sample: ['2600RA', '3901GS', '3939PG', '3902NU', '5200TV']

INPATIENT

CLM_ID
dtype: int64
non-null: 66773
unique: 66705
sample: ['196661176988405', '196201177000368', '196661177015632', '196091176981058', '196261176983265']

CLAIM_KEY
dtype: object
non-null: 66773
unique: 66773
sample: ['196661176988405_1', '196201177000368_1', '196661177015632_1', '196091176981058_1', '196261176983265_1']

DESYNPUF_ID
dtype: object
non-null: 66773
u

In [3]:
# ============================================================
# COMPARE FEATURE COLUMNS ACROSS THE THREE CLAIM DATASETS
# ============================================================

datasets = {
    "OUTPATIENT": outpatient,
    "INPATIENT": inpatient,
    "CARRIER": carrier
}

for name, df in datasets.items():

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    print("Shape:", df.shape)

    print("\nColumns:")
    for i, col in enumerate(df.columns, 1):
        print(f"{i:3}. {col}")

# ============================================================
# FIND COMMON COLUMNS
# ============================================================

out_cols = set(outpatient.columns)
inp_cols = set(inpatient.columns)
car_cols = set(carrier.columns)

print("\n" + "=" * 80)
print("COMMON COLUMNS ACROSS ALL THREE")
print("=" * 80)

common_all = sorted(out_cols & inp_cols & car_cols)

for col in common_all:
    print(col)

print("\nNumber of common columns:", len(common_all))

# ============================================================
# PAIRWISE OVERLAP
# ============================================================

print("\n" + "=" * 80)
print("PAIRWISE COLUMN OVERLAP")
print("=" * 80)

print(
    "Outpatient + Inpatient:",
    len(out_cols & inp_cols)
)

print(
    "Outpatient + Carrier:",
    len(out_cols & car_cols)
)

print(
    "Inpatient + Carrier:",
    len(inp_cols & car_cols)
)

# ============================================================
# UNIQUE COLUMNS BY DATASET
# ============================================================

print("\n" + "=" * 80)
print("OUTPATIENT-ONLY COLUMNS")
print("=" * 80)

for col in sorted(out_cols - inp_cols - car_cols):
    print(col)

print("\n" + "=" * 80)
print("INPATIENT-ONLY COLUMNS")
print("=" * 80)

for col in sorted(inp_cols - out_cols - car_cols):
    print(col)

print("\n" + "=" * 80)
print("CARRIER-ONLY COLUMNS")
print("=" * 80)

for col in sorted(car_cols - out_cols - inp_cols):
    print(col)


OUTPATIENT
Shape: (790790, 57)

Columns:
  1. CLAIM_KEY
  2. DESYNPUF_ID
  3. CLM_ID
  4. SEGMENT
  5. PRVDR_NUM
  6. CLM_PMT_AMT
  7. NCH_PRMRY_PYR_CLM_PD_AMT
  8. NCH_BENE_BLOOD_DDCTBL_LBLTY_AM
  9. NCH_BENE_PTB_DDCTBL_AMT
 10. NCH_BENE_PTB_COINSRNC_AMT
 11. TOTAL_REIMBURSEMENT
 12. CLM_FROM_DT
 13. CLM_THRU_DT
 14. CLAIM_DURATION_DAYS
 15. CLAIM_YEAR
 16. CLAIM_MONTH
 17. DIAGNOSIS_COUNT
 18. PROCEDURE_COUNT
 19. HCPCS_COUNT
 20. HAS_DIAGNOSIS
 21. HAS_PROCEDURE
 22. HAS_HCPCS
 23. HAS_NEGATIVE_PAYMENT
 24. HAS_PRIMARY_PAYER_PAYMENT
 25. IS_SEGMENT_2
 26. HAS_SEGMENT_1_MATCH
 27. BENE_BIRTH_DT
 28. BENE_DEATH_DT
 29. BENE_SEX_IDENT_CD
 30. BENE_RACE_CD
 31. BENE_ESRD_IND
 32. SP_STATE_CODE
 33. BENE_COUNTY_CD
 34. BENE_HI_CVRAGE_TOT_MONS
 35. BENE_SMI_CVRAGE_TOT_MONS
 36. BENE_HMO_CVRAGE_TOT_MONS
 37. PLAN_CVRG_MOS_NUM
 38. SP_ALZHDMTA
 39. SP_CHF
 40. SP_CHRNKIDN
 41. SP_CNCR
 42. SP_COPD
 43. SP_DEPRESSN
 44. SP_DIABETES
 45. SP_ISCHMCHT
 46. SP_OSTEOPRS
 47. SP_RA_OA
 48. SP_STR

In [4]:
# ============================================================
# FINAL ML FEATURE TYPE AUDIT
# ============================================================

datasets = {
    "OUTPATIENT": outpatient,
    "INPATIENT": inpatient,
    "CARRIER": carrier
}

for name, df in datasets.items():

    print("\n" + "=" * 90)
    print(f"{name} ML-RELEVANT NUMERIC / IDENTIFIER AUDIT")
    print("=" * 90)

    # Show columns that are potentially useful for ML
    for col in df.columns:

        if (
            "PAYMENT" in col.upper()
            or "REIMBURSEMENT" in col.upper()
            or "PAY" in col.upper()
            or "COUNT" in col.upper()
            or "RATE" in col.upper()
            or "DURATION" in col.upper()
            or "RATIO" in col.upper()
            or "CHARGE" in col.upper()
            or "COINSURANCE" in col.upper()
            or "DEDUCTIBLE" in col.upper()
            or "COVERAGE" in col.upper()
            or "CVRG" in col.upper()
            or "DIAGNOSIS" in col.upper()
            or "PROCEDURE" in col.upper()
            or "HCPCS" in col.upper()
            or col.upper() in [
                "BENE_SEX_IDENT_CD",
                "BENE_RACE_CD",
                "BENE_ESRD_IND",
                "SP_STATE_CODE",
                "BENE_COUNTY_CD"
            ]
        ):
            print(
                f"{col:45} "
                f"dtype={str(df[col].dtype):12} "
                f"missing={df[col].isna().sum():8}"
            )

    print("\n")


OUTPATIENT ML-RELEVANT NUMERIC / IDENTIFIER AUDIT
TOTAL_REIMBURSEMENT                           dtype=float64      missing=       0
CLAIM_DURATION_DAYS                           dtype=float64      missing=   11253
DIAGNOSIS_COUNT                               dtype=int64        missing=       0
PROCEDURE_COUNT                               dtype=int64        missing=       0
HCPCS_COUNT                                   dtype=int64        missing=       0
HAS_DIAGNOSIS                                 dtype=int64        missing=       0
HAS_PROCEDURE                                 dtype=int64        missing=       0
HAS_HCPCS                                     dtype=int64        missing=       0
HAS_NEGATIVE_PAYMENT                          dtype=int64        missing=       0
HAS_PRIMARY_PAYER_PAYMENT                     dtype=int64        missing=       0
BENE_SEX_IDENT_CD                             dtype=float64      missing=   11565
BENE_RACE_CD                                  d

In [5]:
# ============================================================
# BENEFICIARY FEATURE CONSISTENCY CHECK
# ============================================================

beneficiary_features = [
    "BENE_BIRTH_DT",
    "BENE_DEATH_DT",
    "BENE_SEX_IDENT_CD",
    "BENE_RACE_CD",
    "BENE_ESRD_IND",
    "SP_STATE_CODE",
    "BENE_COUNTY_CD",
    "BENE_HI_CVRAGE_TOT_MONS",
    "BENE_SMI_CVRAGE_TOT_MONS",
    "BENE_HMO_CVRAGE_TOT_MONS",
    "PLAN_CVRG_MOS_NUM",
    "SP_ALZHDMTA",
    "SP_CHF",
    "SP_CHRNKIDN",
    "SP_CNCR",
    "SP_COPD",
    "SP_DEPRESSN",
    "SP_DIABETES",
    "SP_ISCHMCHT",
    "SP_OSTEOPRS",
    "SP_RA_OA",
    "SP_STRKETIA",
    "MEDREIMB_IP",
    "BENRES_IP",
    "PPPYMT_IP",
    "MEDREIMB_OP",
    "BENRES_OP",
    "PPPYMT_OP",
    "MEDREIMB_CAR",
    "BENRES_CAR",
    "PPPYMT_CAR"
]

for col in beneficiary_features:

    print(
        f"{col:35} | "
        f"OUT={str(outpatient[col].dtype):10} | "
        f"INP={str(inpatient[col].dtype):10} | "
        f"CAR={str(carrier[col].dtype):10}"
    )

BENE_BIRTH_DT                       | OUT=object     | INP=object     | CAR=object    
BENE_DEATH_DT                       | OUT=object     | INP=object     | CAR=object    
BENE_SEX_IDENT_CD                   | OUT=float64    | INP=float64    | CAR=int64     
BENE_RACE_CD                        | OUT=float64    | INP=float64    | CAR=int64     
BENE_ESRD_IND                       | OUT=object     | INP=object     | CAR=object    
SP_STATE_CODE                       | OUT=float64    | INP=float64    | CAR=int64     
BENE_COUNTY_CD                      | OUT=float64    | INP=float64    | CAR=int64     
BENE_HI_CVRAGE_TOT_MONS             | OUT=float64    | INP=float64    | CAR=int64     
BENE_SMI_CVRAGE_TOT_MONS            | OUT=float64    | INP=float64    | CAR=int64     
BENE_HMO_CVRAGE_TOT_MONS            | OUT=float64    | INP=float64    | CAR=int64     
PLAN_CVRG_MOS_NUM                   | OUT=float64    | INP=float64    | CAR=int64     
SP_ALZHDMTA                         | OUT=f

In [6]:
# ============================================================
# CHECK DATE TYPES AND BENEFICIARY FEATURE VALUES
# BEFORE BUILDING UNIFIED DATASET
# ============================================================

datasets = {
    "OUTPATIENT": outpatient,
    "INPATIENT": inpatient,
    "CARRIER": carrier
}

for name, df in datasets.items():

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    for col in ["BENE_BIRTH_DT", "BENE_DEATH_DT"]:
        print(f"\n{col}")
        print("dtype:", df[col].dtype)
        print("missing:", df[col].isna().sum())
        print("sample:")
        print(df[col].dropna().head(5).tolist())

    print("\nCLAIM YEAR")
    year_col = "CLAIM_YEAR" if "CLAIM_YEAR" in df.columns else "claim_year"
    print(df[year_col].value_counts(dropna=False).sort_index())

# ============================================================
# CHECK BENEFICIARY CATEGORICAL VALUE DISTRIBUTIONS
# ============================================================

categorical_beneficiary_cols = [
    "BENE_SEX_IDENT_CD",
    "BENE_RACE_CD",
    "BENE_ESRD_IND",
    "SP_STATE_CODE"
]

for col in categorical_beneficiary_cols:

    print("\n" + "=" * 80)
    print(col)
    print("=" * 80)

    for name, df in datasets.items():

        if col in df.columns:

            print(f"\n{name}:")
            print(
                df[col]
                .value_counts(dropna=False)
                .head(20)
            )


OUTPATIENT

BENE_BIRTH_DT
dtype: object
missing: 11565
sample:
['1923-05-01', '1943-01-01', '1943-01-01', '1936-09-01', '1936-08-01']

BENE_DEATH_DT
dtype: object
missing: 784803
sample:
['2008-07-01', '2008-07-01', '2008-07-01', '2009-09-01', '2009-09-01']

CLAIM YEAR
CLAIM_YEAR
2007.0       312
2008.0    282896
2009.0    322358
2010.0    173971
NaN        11253
Name: count, dtype: int64

INPATIENT

BENE_BIRTH_DT
dtype: object
missing: 292
sample:
['1923-05-01', '1943-01-01', '1943-01-01', '1943-01-01', '1943-01-01']

BENE_DEATH_DT
dtype: object
missing: 66248
sample:
['2009-09-01', '2010-06-01', '2010-08-01', '2008-12-01', '2008-12-01']

CLAIM YEAR
CLAIM_YEAR
2007.0      224
2008.0    27678
2009.0    25231
2010.0    13572
NaN          68
Name: count, dtype: int64

CARRIER

BENE_BIRTH_DT
dtype: object
missing: 0
sample:
['1923-05-01', '1923-05-01', '1923-05-01', '1923-05-01', '1923-05-01']

BENE_DEATH_DT
dtype: object
missing: 4704863
sample:
['2008-09-01', '2008-09-01', '2008-09-01'

In [7]:
# ============================================================
# FINAL NUMERICAL RANGE AUDIT BEFORE ML FEATURE ENGINEERING
# ============================================================

datasets = {
    "OUTPATIENT": outpatient,
    "INPATIENT": inpatient,
    "CARRIER": carrier
}

for name, df in datasets.items():

    print("\n" + "=" * 90)
    print(f"{name} NUMERICAL RANGE AUDIT")
    print("=" * 90)

    numeric_cols = df.select_dtypes(include=["number"]).columns

    summary = df[numeric_cols].describe().T[
        ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
    ]

    print(summary.to_string())


OUTPATIENT NUMERICAL RANGE AUDIT
                                   count          mean           std           min           25%           50%           75%           max
CLM_ID                          790790.0  5.425026e+14  2.858482e+11  5.420123e+14  5.422523e+14  5.425023e+14  5.427523e+14  5.429923e+14
SEGMENT                         790790.0  1.014230e+00  1.184382e-01  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00  2.000000e+00
CLM_PMT_AMT                     790790.0  2.839246e+02  5.713928e+02 -1.000000e+02  4.000000e+01  8.000000e+01  2.000000e+02  3.300000e+03
NCH_PRMRY_PYR_CLM_PD_AMT        790790.0  1.023976e+01  2.346684e+02  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  1.400000e+04
NCH_BENE_BLOOD_DDCTBL_LBLTY_AM  790790.0  1.289849e-02  2.315506e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  8.000000e+02
NCH_BENE_PTB_DDCTBL_AMT         790790.0  2.825466e+00  1.559652e+01  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  2.0000

In [8]:
# ============================================================
# NEGATIVE / ZERO / EXTREME VALUE CHECK
# ============================================================

checks = {
    "OUTPATIENT": {
        "df": outpatient,
        "columns": [
            "CLM_PMT_AMT",
            "TOTAL_REIMBURSEMENT",
            "CLAIM_DURATION_DAYS",
            "DIAGNOSIS_COUNT",
            "PROCEDURE_COUNT",
            "HCPCS_COUNT"
        ]
    },

    "INPATIENT": {
        "df": inpatient,
        "columns": [
            "CLM_PMT_AMT",
            "CLM_UTLZTN_DAY_CNT",
            "CLAIM_DURATION_DAYS",
            "DIAGNOSIS_COUNT",
            "PROCEDURE_COUNT"
        ]
    },

    "CARRIER": {
        "df": carrier,
        "columns": [
            "total_claim_payment_amt",
            "total_allowed_charge_amt",
            "total_deductible_amt",
            "total_coinsurance_amt",
            "total_primary_payer_paid_amt",
            "avg_payment_per_line",
            "payment_to_allowed_ratio",
            "line_count",
            "unique_hcpcs_count",
            "max_line_payment",
            "diagnosis_count",
            "unique_diagnosis_count",
            "provider_claim_volume",
            "provider_avg_claim_payment"
        ]
    }
}

for name, info in checks.items():

    df = info["df"]

    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    for col in info["columns"]:

        s = df[col]

        print(
            f"{col:40} "
            f"negative={int((s < 0).sum()):8} "
            f"zero={int((s == 0).sum()):8} "
            f"missing={int(s.isna().sum()):8} "
            f"max={s.max()}"
        )


OUTPATIENT
CLM_PMT_AMT                              negative=    2566 zero=   30105 missing=       0 max=3300.0
TOTAL_REIMBURSEMENT                      negative=    1393 zero=    9843 missing=       0 max=18400.0
CLAIM_DURATION_DAYS                      negative=       0 zero=       0 missing=   11253 max=21.0
DIAGNOSIS_COUNT                          negative=       0 zero=    5606 missing=       0 max=10
PROCEDURE_COUNT                          negative=       0 zero=  790590 missing=       0 max=6
HCPCS_COUNT                              negative=       0 zero=   37360 missing=       0 max=44

INPATIENT
CLM_PMT_AMT                              negative=      55 zero=    2160 missing=       0 max=57000.0
CLM_UTLZTN_DAY_CNT                       negative=       0 zero=    2266 missing=      68 max=136.0
CLAIM_DURATION_DAYS                      negative=       0 zero=       0 missing=      68 max=36.0
DIAGNOSIS_COUNT                          negative=       0 zero=      68 missing=   

In [11]:
import pandas as pd
from pathlib import Path

BASE = Path("../data/processed/primary")

outpatient_file = BASE / "outpatient_claims_with_beneficiary_features.csv"
inpatient_file = BASE / "inpatient_claims_fully_enriched.csv"
carrier_file = BASE / "carrier_claims_with_beneficiary_features.csv"

output_file = BASE / "unified_claims_ml_base.csv"


# ============================================================
# 1. LOAD OUTPATIENT
# ============================================================

print("Loading outpatient...")

outpatient = pd.read_csv(outpatient_file)

outpatient["CLAIM_TYPE"] = "OUTPATIENT"

# Outpatient already has CLAIM_YEAR and CLAIM_MONTH
print("Outpatient:", outpatient.shape)


# ============================================================
# 2. LOAD INPATIENT
# ============================================================

print("Loading inpatient...")

inpatient = pd.read_csv(inpatient_file)

inpatient["CLAIM_TYPE"] = "INPATIENT"

# Inpatient has CLAIM_YEAR but NOT CLAIM_MONTH.
# We intentionally leave CLAIM_MONTH as missing.
inpatient["CLAIM_MONTH"] = pd.NA

print("Inpatient:", inpatient.shape)


# ============================================================
# 3. LOAD CARRIER
# ============================================================

print("Loading carrier...")

carrier = pd.read_csv(carrier_file)

carrier["CLAIM_TYPE"] = "CARRIER"

# Carrier uses lowercase claim_year / claim_month.
carrier["CLAIM_YEAR"] = carrier["claim_year"]
carrier["CLAIM_MONTH"] = carrier["claim_month"]

print("Carrier:", carrier.shape)


# ============================================================
# 4. STANDARDIZE CLAIM IDENTIFIER
# ============================================================

# Outpatient and inpatient already have CLAIM_KEY.
# Carrier does not.

carrier["CLAIM_KEY"] = (
    carrier["CLM_ID"].astype(str) + "_CARRIER"
)


# ============================================================
# 5. ALIGN ALL COLUMNS
# ============================================================

all_columns = sorted(
    set(outpatient.columns)
    | set(inpatient.columns)
    | set(carrier.columns)
)

print("\nTotal unified columns:", len(all_columns))


outpatient = outpatient.reindex(columns=all_columns)
inpatient = inpatient.reindex(columns=all_columns)
carrier = carrier.reindex(columns=all_columns)


# ============================================================
# 6. COMBINE
# ============================================================

print("\nCombining datasets...")

unified = pd.concat(
    [
        outpatient,
        inpatient,
        carrier
    ],
    ignore_index=True
)


# ============================================================
# 7. VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("UNIFIED CLAIM DATASET")
print("=" * 80)

print("\nShape:")
print(unified.shape)

print("\nExpected rows:")
print(790790 + 66773 + 4741335)

print("\nClaim type distribution:")
print(unified["CLAIM_TYPE"].value_counts())

print("\nUnique beneficiaries:")
print(unified["DESYNPUF_ID"].nunique())

print("\nUnique CLAIM_KEY:")
print(unified["CLAIM_KEY"].nunique())

print("\nDuplicate CLAIM_KEY:")
print(unified["CLAIM_KEY"].duplicated().sum())

print("\nMissing CLAIM_TYPE:")
print(unified["CLAIM_TYPE"].isna().sum())

print("\nCLAIM_YEAR distribution:")
print(
    unified.groupby(
        ["CLAIM_TYPE", "CLAIM_YEAR"],
        dropna=False
    ).size()
)


# ============================================================
# 8. SAVE
# ============================================================

unified.to_csv(output_file, index=False)

print("\n" + "=" * 80)
print("SAVED")
print("=" * 80)

print(output_file)
print("Final shape:", unified.shape)

Loading outpatient...
Outpatient: (790790, 58)
Loading inpatient...
Inpatient: (66773, 94)
Loading carrier...
Carrier: (4741335, 55)

Total unified columns: 122

Combining datasets...


C:\Users\Admin\AppData\Local\Temp\ipykernel_17048\70636505.py:97: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  unified = pd.concat(



UNIFIED CLAIM DATASET

Shape:
(5598898, 122)

Expected rows:
5598898

Claim type distribution:
CLAIM_TYPE
CARRIER       4741335
OUTPATIENT     790790
INPATIENT       66773
Name: count, dtype: int64

Unique beneficiaries:
99226

Unique CLAIM_KEY:
5598898

Duplicate CLAIM_KEY:
0

Missing CLAIM_TYPE:
0

CLAIM_YEAR distribution:
CLAIM_TYPE  CLAIM_YEAR
CARRIER     2008.0        1715402
            2009.0        1862973
            2010.0        1162960
INPATIENT   2007.0            224
            2008.0          27678
            2009.0          25231
            2010.0          13572
            NaN                68
OUTPATIENT  2007.0            312
            2008.0         282896
            2009.0         322358
            2010.0         173971
            NaN             11253
dtype: int64

SAVED
..\data\processed\primary\unified_claims_ml_base.csv
Final shape: (5598898, 122)


In [14]:
import pandas as pd
from pathlib import Path
from collections import defaultdict

BASE = Path("../data/processed/primary")
file = BASE / "unified_claims_ml_base.csv"

CHUNK_SIZE = 100_000

print("=" * 90)
print("UNIFIED DATASET - MEMORY SAFE ML FEATURE AUDIT")
print("=" * 90)

# ------------------------------------------------------------
# Read header only
# ------------------------------------------------------------

header = pd.read_csv(file, nrows=0)

columns = header.columns.tolist()

print("Number of columns:", len(columns))
print("Columns loaded from header:", len(columns))


# ------------------------------------------------------------
# Initialize statistics
# ------------------------------------------------------------

missing_counts = pd.Series(0, index=columns, dtype="int64")
unique_values = defaultdict(set)

dtype_seen = {}

row_count = 0


# ------------------------------------------------------------
# Process CSV in chunks
# ------------------------------------------------------------

print("\nProcessing file in chunks...")

for i, chunk in enumerate(
    pd.read_csv(
        file,
        chunksize=CHUNK_SIZE,
        low_memory=True
    )
):

    row_count += len(chunk)

    # Missing values
    missing_counts += chunk.isna().sum()

    # Record dtypes
    for c in chunk.columns:
        dtype_seen[c] = str(chunk[c].dtype)

    # Unique values
    #
    # We only need exact unique values for relatively
    # low-cardinality columns. For high-cardinality columns,
    # storing millions of values would defeat the purpose
    # of the memory-safe approach.
    for c in chunk.columns:

        nunique_chunk = chunk[c].nunique(dropna=True)

        if nunique_chunk <= 1000:

            vals = chunk[c].dropna().unique()

            # Protect against very large categorical objects
            if len(unique_values[c]) <= 1000:

                for v in vals:
                    unique_values[c].add(v)

                if len(unique_values[c]) > 1000:
                    unique_values[c] = set(
                        list(unique_values[c])[:1001]
                    )

    if (i + 1) % 10 == 0:
        print(
            f"Processed approximately "
            f"{row_count:,} rows..."
        )


print("\nFinished processing.")
print("Total rows:", row_count)


# ------------------------------------------------------------
# Build inventory
# ------------------------------------------------------------

inventory = pd.DataFrame({
    "column": columns,
    "dtype": [
        dtype_seen.get(c, "unknown")
        for c in columns
    ],
    "missing_count": [
        missing_counts[c]
        for c in columns
    ]
})

inventory["missing_pct"] = (
    inventory["missing_count"]
    / row_count
    * 100
)

inventory["unique_count"] = [
    len(unique_values[c])
    if c in unique_values
    else -1
    for c in columns
]

inventory["unique_pct"] = (
    inventory["unique_count"]
    / row_count
    * 100
)


# ------------------------------------------------------------
# Print basic inventory
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("COLUMN INVENTORY")
print("=" * 90)

print(
    inventory.to_string(index=False)
)


# ------------------------------------------------------------
# Identifier columns
# ------------------------------------------------------------

identifier_cols = [
    "CLAIM_KEY",
    "CLM_ID",
    "DESYNPUF_ID"
]

print("\n" + "=" * 90)
print("IDENTIFIERS")
print("=" * 90)

print(
    inventory[
        inventory["column"].isin(identifier_cols)
    ].to_string(index=False)
)


# ------------------------------------------------------------
# Date columns
# ------------------------------------------------------------

date_cols = [
    c for c in columns
    if (
        c.endswith("_DT")
        or "DATE" in c.upper()
    )
]

print("\n" + "=" * 90)
print("DATE / DATE-LIKE COLUMNS")
print("=" * 90)

print(
    inventory[
        inventory["column"].isin(date_cols)
    ].to_string(index=False)
)


# ------------------------------------------------------------
# Potential categorical/code columns
# ------------------------------------------------------------

categorical_candidates = [
    c for c in columns
    if (
        "ICD9" in c
        or "HCPCS" in c
        or "DRG" in c
        or c in [
            "CLAIM_TYPE",
            "BENE_ESRD_IND",
            "BENE_SEX_IDENT_CD",
            "BENE_RACE_CD",
            "SP_STATE_CODE",
            "BENE_COUNTY_CD",
            "PRVDR_NUM",
            "primary_provider_npi"
        ]
    )
]

print("\n" + "=" * 90)
print("CATEGORICAL / CODE CANDIDATES")
print("=" * 90)

print(
    inventory[
        inventory["column"].isin(
            categorical_candidates
        )
    ].to_string(index=False)
)


# ------------------------------------------------------------
# Very sparse columns
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("VERY SPARSE COLUMNS (>50% MISSING)")
print("=" * 90)

sparse = inventory[
    inventory["missing_pct"] > 50
].sort_values(
    "missing_pct",
    ascending=False
)

print(
    sparse.to_string(index=False)
)


# ------------------------------------------------------------
# Constant columns
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CONSTANT / VERY LOW CARDINALITY COLUMNS")
print("=" * 90)

print(
    inventory[
        (inventory["unique_count"] >= 0)
        & (inventory["unique_count"] <= 1)
    ].to_string(index=False)
)


# ------------------------------------------------------------
# Save inventory
# ------------------------------------------------------------

inventory_file = (
    BASE /
    "unified_claims_ml_feature_inventory.csv"
)

inventory.to_csv(
    inventory_file,
    index=False
)

print("\n" + "=" * 90)
print("INVENTORY SAVED")
print("=" * 90)

print(inventory_file)
print("Inventory shape:", inventory.shape)

UNIFIED DATASET - MEMORY SAFE ML FEATURE AUDIT
Number of columns: 122
Columns loaded from header: 122

Processing file in chunks...


C:\Users\Admin\AppData\Local\Temp\ipykernel_17048\3137767240.py:44: DtypeWarning: Columns (0,26,27,43,44,45,46,47,48,49,50,51,52,54,55,56,57,58,67) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(
C:\Users\Admin\AppData\Local\Temp\ipykernel_17048\3137767240.py:44: DtypeWarning: Columns (0,26,27,28,32,43,44,45,46,47,48,49,50,51,52,54,55,56,57,58,67,85,112) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(


Processed approximately 1,000,000 rows...
Processed approximately 2,000,000 rows...


C:\Users\Admin\AppData\Local\Temp\ipykernel_17048\3137767240.py:44: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(
C:\Users\Admin\AppData\Local\Temp\ipykernel_17048\3137767240.py:44: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(
C:\Users\Admin\AppData\Local\Temp\ipykernel_17048\3137767240.py:44: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(
C:\Users\Admin\AppData\Local\Temp\ipykernel_17048\3137767240.py:44: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(


Processed approximately 3,000,000 rows...


...

Processed approximately 4,000,000 rows...


C:\Users\Admin\AppData\Local\Temp\ipykernel_17048\3137767240.py:44: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(


Processed approximately 5,000,000 rows...


C:\Users\Admin\AppData\Local\Temp\ipykernel_17048\3137767240.py:44: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(
C:\Users\Admin\AppData\Local\Temp\ipykernel_17048\3137767240.py:44: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(
C:\Users\Admin\AppData\Local\Temp\ipykernel_17048\3137767240.py:44: DtypeWarning: Columns (112) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(



Finished processing.
Total rows: 5598898

COLUMN INVENTORY
                          column   dtype  missing_count  missing_pct  unique_count  unique_pct
             ADMTNG_ICD9_DGNS_CD float64        5532724    98.818089             0    0.000000
                    AT_PHYSN_NPI float64        5532798    98.819411             0    0.000000
              AVG_CLAIM_DURATION float64        5532125    98.807390           979    0.017486
             AVG_DIAGNOSIS_COUNT float64        5532125    98.807390           716    0.012788
                     AVG_PAYMENT float64        5532125    98.807390             0    0.000000
             AVG_PROCEDURE_COUNT float64        5532125    98.807390           699    0.012485
                   BENE_BIRTH_DT  object          11857     0.211774           900    0.016075
                  BENE_COUNTY_CD float64          11857     0.211774           307    0.005483
                   BENE_DEATH_DT  object        5555914    99.232277            36   

In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# PATHS
# ============================================================

BASE = Path("../data/processed/primary")

INPUT_FILE = BASE / "unified_claims_ml_base.csv"
OUTPUT_FILE = BASE / "claims_ml_features.csv"

# ============================================================
# FEATURES FOR THE FIRST ML VERSION
# ============================================================

ID_COLS = [
    "CLAIM_KEY",
    "CLM_ID",
    "DESYNPUF_ID",
    "CLAIM_TYPE"
]

# Features that can be useful across claim types
UNIVERSAL_FEATURES = [
    "CLAIM_YEAR",
    "CLAIM_MONTH",

    # beneficiary characteristics
    "BENE_SEX_IDENT_CD",
    "BENE_RACE_CD",
    "BENE_ESRD_IND",
    "SP_STATE_CODE",
    "BENE_COUNTY_CD",
    "BENE_HI_CVRAGE_TOT_MONS",
    "BENE_SMI_CVRAGE_TOT_MONS",
    "BENE_HMO_CVRAGE_TOT_MONS",
    "PLAN_CVRG_MOS_NUM",

    # chronic conditions
    "SP_ALZHDMTA",
    "SP_CHF",
    "SP_CHRNKIDN",
    "SP_CNCR",
    "SP_COPD",
    "SP_DEPRESSN",
    "SP_DIABETES",
    "SP_ISCHMCHT",
    "SP_OSTEOPRS",
    "SP_RA_OA",
    "SP_STRKETIA",

    # beneficiary reimbursement
    "MEDREIMB_IP",
    "BENRES_IP",
    "PPPYMT_IP",
    "MEDREIMB_OP",
    "BENRES_OP",
    "PPPYMT_OP",
    "MEDREIMB_CAR",
    "BENRES_CAR",
    "PPPYMT_CAR",
]

# Outpatient / inpatient claim-level features
MEDICAL_CLAIM_FEATURES = [
    "CLM_PMT_AMT",
    "NCH_PRMRY_PYR_CLM_PD_AMT",
    "TOTAL_REIMBURSEMENT",
    "CLAIM_DURATION_DAYS",
    "DIAGNOSIS_COUNT",
    "PROCEDURE_COUNT",
    "HCPCS_COUNT",
    "HAS_DIAGNOSIS",
    "HAS_PROCEDURE",
    "HAS_HCPCS",
    "HAS_NEGATIVE_PAYMENT",
    "HAS_PRIMARY_PAYER_PAYMENT",
    "IS_SEGMENT_2",
    "HAS_SEGMENT_1_MATCH",

    # inpatient-specific
    "CLM_UTLZTN_DAY_CNT",
]

# Inpatient provider behavior
INPATIENT_PROVIDER_FEATURES = [
    "CLAIM_COUNT",
    "UNIQUE_CLAIM_COUNT",
    "TOTAL_PAYMENT",
    "AVG_PAYMENT",
    "MEDIAN_PAYMENT",
    "MAX_PAYMENT",
    "AVG_CLAIM_DURATION",
    "MAX_CLAIM_DURATION",
    "AVG_DIAGNOSIS_COUNT",
    "AVG_PROCEDURE_COUNT",
    "NEGATIVE_PAYMENT_COUNT",
    "PRIMARY_PAYER_PAYMENT_COUNT",
    "NEGATIVE_PAYMENT_RATE",
    "PRIMARY_PAYER_PAYMENT_RATE",
    "UNIQUE_BENEFICIARIES",
    "CLAIMS_PER_BENEFICIARY",
    "PAYMENT_STD",
    "CLAIM_DURATION_STD",
]

# Carrier behavior
CARRIER_FEATURES = [
    "total_claim_payment_amt",
    "total_allowed_charge_amt",
    "total_deductible_amt",
    "total_coinsurance_amt",
    "total_primary_payer_paid_amt",
    "avg_payment_per_line",
    "payment_to_allowed_ratio",
    "line_count",
    "unique_hcpcs_count",
    "max_line_payment",
    "diagnosis_count",
    "unique_diagnosis_count",
    "distinct_provider_count_on_claim",
    "provider_claim_volume",
    "provider_avg_claim_payment",
]

FEATURE_COLS = (
    UNIVERSAL_FEATURES
    + MEDICAL_CLAIM_FEATURES
    + INPATIENT_PROVIDER_FEATURES
    + CARRIER_FEATURES
)

# Remove duplicates while preserving order
FEATURE_COLS = list(dict.fromkeys(FEATURE_COLS))

USE_COLS = ID_COLS + FEATURE_COLS

print("Requested feature columns:", len(FEATURE_COLS))
print("Total columns to read:", len(USE_COLS))

# ============================================================
# CHECK WHICH COLUMNS ACTUALLY EXIST
# ============================================================

header = pd.read_csv(INPUT_FILE, nrows=0)
available = set(header.columns)

missing_requested = [c for c in USE_COLS if c not in available]

if missing_requested:
    print("\nWARNING - columns not found:")
    for c in missing_requested:
        print("  ", c)

USE_COLS = [c for c in USE_COLS if c in available]

print("\nColumns actually being loaded:", len(USE_COLS))

# ============================================================
# CHUNKED PROCESSING
# ============================================================

CHUNK_SIZE = 250_000

first_chunk = True
total_rows = 0

if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()

for i, chunk in enumerate(
    pd.read_csv(
        INPUT_FILE,
        usecols=USE_COLS,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )
):

    # --------------------------------------------------------
    # Ensure numeric feature columns are numeric
    # --------------------------------------------------------

    feature_cols_in_chunk = [
        c for c in FEATURE_COLS
        if c in chunk.columns
    ]

    for col in feature_cols_in_chunk:
        if col == "BENE_ESRD_IND":
            # Convert 0/Y into numeric representation
            chunk[col] = (
                chunk[col]
                .astype("string")
                .str.upper()
                .map({"0": 0, "Y": 1})
            )
        else:
            chunk[col] = pd.to_numeric(
                chunk[col],
                errors="coerce"
            )

    # --------------------------------------------------------
    # Claim type indicators
    # --------------------------------------------------------

    chunk["IS_OUTPATIENT"] = (
        chunk["CLAIM_TYPE"] == "OUTPATIENT"
    ).astype("int8")

    chunk["IS_INPATIENT"] = (
        chunk["CLAIM_TYPE"] == "INPATIENT"
    ).astype("int8")

    chunk["IS_CARRIER"] = (
        chunk["CLAIM_TYPE"] == "CARRIER"
    ).astype("int8")

    # --------------------------------------------------------
    # Missingness indicators
    #
    # Important because missingness is partly caused by
    # claim type. These indicators let the model distinguish
    # "not applicable" from potentially unusual missing data.
    # --------------------------------------------------------

    for col in [
        "CLM_PMT_AMT",
        "TOTAL_REIMBURSEMENT",
        "CLAIM_DURATION_DAYS",
        "DIAGNOSIS_COUNT",
        "PROCEDURE_COUNT",
        "HCPCS_COUNT",

        "total_claim_payment_amt",
        "total_allowed_charge_amt",
        "payment_to_allowed_ratio",

        "CLAIM_COUNT",
        "AVG_PAYMENT",
        "PAYMENT_STD",
    ]:

        if col in chunk.columns:
            chunk[f"{col}_MISSING"] = (
                chunk[col].isna()
            ).astype("int8")

    # --------------------------------------------------------
    # Numeric missing values
    #
    # For now use median within each claim type.
    # This avoids letting the huge carrier population dominate
    # the inpatient/outpatient values.
    # --------------------------------------------------------

    numeric_cols = [
        c for c in chunk.columns
        if c not in ID_COLS
    ]

    numeric_cols = chunk[numeric_cols].select_dtypes(
        include=[np.number]
    ).columns

    for claim_type in ["OUTPATIENT", "INPATIENT", "CARRIER"]:

        mask = chunk["CLAIM_TYPE"] == claim_type

        if not mask.any():
            continue

        for col in numeric_cols:
            if chunk.loc[mask, col].isna().any():

                median_value = chunk.loc[
                    mask, col
                ].median()

                if pd.isna(median_value):
                    median_value = 0

                chunk.loc[mask, col] = (
                    chunk.loc[mask, col]
                    .fillna(median_value)
                )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    chunk.to_csv(
        OUTPUT_FILE,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False
    total_rows += len(chunk)

    print(
        f"Processed {total_rows:,} rows..."
    )

print("\n" + "=" * 80)
print("ML FEATURE DATASET CREATED")
print("=" * 80)

print("Rows:", total_rows)
print("Saved:", OUTPUT_FILE)

Requested feature columns: 79
Total columns to read: 83

Columns actually being loaded: 83


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\A

Processed 250,000 rows...


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\A

Processed 500,000 rows...


...

Processed 5,250,000 rows...


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\A

Processed 5,500,000 rows...


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\A

Processed 5,598,898 rows...

ML FEATURE DATASET CREATED
Rows: 5598898
Saved: ..\data\processed\primary\claims_ml_features.csv


In [16]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../data/processed/primary")
FILE = BASE / "claims_ml_features.csv"

print("=" * 90)
print("CLAIMS ML FEATURE DATASET VALIDATION")
print("=" * 90)

# ------------------------------------------------------------
# Read in chunks because the file is large
# ------------------------------------------------------------

total_rows = 0
duplicate_claim_keys = 0
missing_counts = None
inf_counts = None
claim_type_counts = {}
feature_dtypes = None

for chunk in pd.read_csv(FILE, chunksize=250_000, low_memory=False):

    total_rows += len(chunk)

    # Claim key duplicates
    dup_count = chunk["CLAIM_KEY"].duplicated().sum()
    duplicate_claim_keys += dup_count

    # Missing values
    current_missing = chunk.isna().sum()

    if missing_counts is None:
        missing_counts = current_missing
    else:
        missing_counts = missing_counts.add(
            current_missing,
            fill_value=0
        )

    # Infinite values
    numeric = chunk.select_dtypes(include=[np.number])

    current_inf = np.isinf(numeric).sum()

    if inf_counts is None:
        inf_counts = current_inf
    else:
        inf_counts = inf_counts.add(
            current_inf,
            fill_value=0
        )

    # Claim type
    if "CLAIM_TYPE" in chunk.columns:
        counts = chunk["CLAIM_TYPE"].value_counts()

        for claim_type, count in counts.items():
            claim_type_counts[claim_type] = (
                claim_type_counts.get(claim_type, 0) + count
            )

    if feature_dtypes is None:
        feature_dtypes = chunk.dtypes

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("\nRows:", f"{total_rows:,}")

print("\nExpected rows:", f"{5_598_898:,}")

print(
    "Row count correct:",
    total_rows == 5_598_898
)

print("\nDuplicate CLAIM_KEY found:")
print(duplicate_claim_keys)

print("\nClaim type distribution:")
print(pd.Series(claim_type_counts))

print("\nMissing values:")
print(
    missing_counts[
        missing_counts > 0
    ].sort_values(ascending=False)
)

print("\nInfinite values:")
print(
    inf_counts[
        inf_counts > 0
    ].sort_values(ascending=False)
)

print("\nNumber of columns:", len(feature_dtypes))

print("\nFeature columns:")
for col in feature_dtypes.index:
    print(col)

print("\n" + "=" * 90)
print("VALIDATION COMPLETE")
print("=" * 90)

CLAIMS ML FEATURE DATASET VALIDATION

Rows: 5,598,898

Expected rows: 5,598,898
Row count correct: True

Duplicate CLAIM_KEY found:
0

Claim type distribution:
OUTPATIENT     790790
CARRIER       4741335
INPATIENT       66773
dtype: int64

Missing values:
Series([], dtype: int64)

Infinite values:
Series([], dtype: int64)

Number of columns: 98

Feature columns:
AVG_CLAIM_DURATION
AVG_DIAGNOSIS_COUNT
AVG_PAYMENT
AVG_PROCEDURE_COUNT
BENE_COUNTY_CD
BENE_ESRD_IND
BENE_HI_CVRAGE_TOT_MONS
BENE_HMO_CVRAGE_TOT_MONS
BENE_RACE_CD
BENE_SEX_IDENT_CD
BENE_SMI_CVRAGE_TOT_MONS
BENRES_CAR
BENRES_IP
BENRES_OP
CLAIMS_PER_BENEFICIARY
CLAIM_COUNT
CLAIM_DURATION_DAYS
CLAIM_DURATION_STD
CLAIM_KEY
CLAIM_MONTH
CLAIM_TYPE
CLAIM_YEAR
CLM_ID
CLM_PMT_AMT
CLM_UTLZTN_DAY_CNT
DESYNPUF_ID
DIAGNOSIS_COUNT
HAS_DIAGNOSIS
HAS_HCPCS
HAS_NEGATIVE_PAYMENT
HAS_PRIMARY_PAYER_PAYMENT
HAS_PROCEDURE
HAS_SEGMENT_1_MATCH
HCPCS_COUNT
IS_SEGMENT_2
MAX_CLAIM_DURATION
MAX_PAYMENT
MEDIAN_PAYMENT
MEDREIMB_CAR
MEDREIMB_IP
MEDREIMB_OP
NC

In [17]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../data/processed/primary")
FILE = BASE / "claims_ml_features.csv"

print("=" * 100)
print("ML FEATURE QUALITY AUDIT")
print("=" * 100)

# ------------------------------------------------------------
# Read only the first chunk for structural inspection
# ------------------------------------------------------------

df = pd.read_csv(
    FILE,
    nrows=250_000,
    low_memory=False
)

# ------------------------------------------------------------
# Columns that must NOT be model features
# ------------------------------------------------------------

ID_COLS = [
    "CLAIM_KEY",
    "CLM_ID",
    "DESYNPUF_ID",
]

TARGET_IDENTIFIER = [
    "CLAIM_TYPE"
]

# ------------------------------------------------------------
# Candidate numerical features
# ------------------------------------------------------------

candidate_features = [
    c for c in df.columns
    if c not in ID_COLS + TARGET_IDENTIFIER
]

numeric_features = df[candidate_features].select_dtypes(
    include=[np.number]
).columns.tolist()

print("\nTotal columns:", len(df.columns))
print("Candidate model columns:", len(candidate_features))
print("Numeric model features:", len(numeric_features))

# ------------------------------------------------------------
# Constant features
# ------------------------------------------------------------

constant_features = []

for col in numeric_features:
    if df[col].nunique(dropna=False) <= 1:
        constant_features.append(col)

print("\n" + "=" * 100)
print("CONSTANT FEATURES")
print("=" * 100)

if constant_features:
    for col in constant_features:
        print(col)
else:
    print("None")

# ------------------------------------------------------------
# Low-variance / near-constant features
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("FEATURE DISTRIBUTIONS")
print("=" * 100)

summary = pd.DataFrame({
    "dtype": df[numeric_features].dtypes.astype(str),
    "unique": df[numeric_features].nunique(),
    "mean": df[numeric_features].mean(),
    "std": df[numeric_features].std(),
    "min": df[numeric_features].min(),
    "median": df[numeric_features].median(),
    "max": df[numeric_features].max()
})

print(summary.to_string())

# ------------------------------------------------------------
# Highly skewed features
# ------------------------------------------------------------

skewness = df[numeric_features].skew().sort_values(
    ascending=False
)

print("\n" + "=" * 100)
print("MOST SKEWED FEATURES")
print("=" * 100)

print(skewness.head(25).to_string())

# ------------------------------------------------------------
# Very high cardinality numerical features
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("HIGH-CARDINALITY NUMERICAL FEATURES")
print("=" * 100)

cardinality = (
    df[numeric_features]
    .nunique()
    .sort_values(ascending=False)
)

print(cardinality.head(25).to_string())

# ------------------------------------------------------------
# Claim type distribution
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CLAIM TYPE")
print("=" * 100)

print(df["CLAIM_TYPE"].value_counts())

print("\n" + "=" * 100)
print("AUDIT COMPLETE")
print("=" * 100)

ML FEATURE QUALITY AUDIT

Total columns: 98
Candidate model columns: 94
Numeric model features: 94

CONSTANT FEATURES
AVG_CLAIM_DURATION
AVG_DIAGNOSIS_COUNT
AVG_PAYMENT
AVG_PROCEDURE_COUNT
CLAIMS_PER_BENEFICIARY
CLAIM_COUNT
CLAIM_DURATION_STD
CLM_UTLZTN_DAY_CNT
MAX_CLAIM_DURATION
MAX_PAYMENT
MEDIAN_PAYMENT
NEGATIVE_PAYMENT_COUNT
NEGATIVE_PAYMENT_RATE
PAYMENT_STD
PRIMARY_PAYER_PAYMENT_COUNT
PRIMARY_PAYER_PAYMENT_RATE
TOTAL_PAYMENT
UNIQUE_BENEFICIARIES
UNIQUE_CLAIM_COUNT
avg_payment_per_line
diagnosis_count
distinct_provider_count_on_claim
line_count
max_line_payment
payment_to_allowed_ratio
provider_avg_claim_payment
provider_claim_volume
total_allowed_charge_amt
total_claim_payment_amt
total_coinsurance_amt
total_deductible_amt
total_primary_payer_paid_amt
unique_diagnosis_count
unique_hcpcs_count
IS_OUTPATIENT
IS_INPATIENT
IS_CARRIER
CLM_PMT_AMT_MISSING
TOTAL_REIMBURSEMENT_MISSING
DIAGNOSIS_COUNT_MISSING
PROCEDURE_COUNT_MISSING
HCPCS_COUNT_MISSING
total_claim_payment_amt_MISSING
total

In [18]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../data/processed/primary")
FILE = BASE / "claims_ml_features.csv"

CHUNK_SIZE = 250_000

# ============================================================
# FEATURES WE DO NOT MODEL DIRECTLY
# ============================================================

ID_COLS = [
    "CLAIM_KEY",
    "CLM_ID",
    "DESYNPUF_ID",
    "CLAIM_TYPE"
]

# ============================================================
# FIRST: GET NUMERIC FEATURE LIST FROM HEADER
# ============================================================

header = pd.read_csv(FILE, nrows=0)

numeric_candidates = [
    c for c in header.columns
    if c not in ID_COLS
]

# ============================================================
# STORAGE
# ============================================================

claim_types = [
    "OUTPATIENT",
    "INPATIENT",
    "CARRIER"
]

stats = {
    claim_type: {}
    for claim_type in claim_types
}

row_counts = {
    claim_type: 0
    for claim_type in claim_types
}

# ============================================================
# PROCESS COMPLETE FILE
# ============================================================

for chunk_number, chunk in enumerate(
    pd.read_csv(
        FILE,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    print(
        f"Scanning chunk {chunk_number}..."
    )

    # --------------------------------------------------------
    # Process each claim type separately
    # --------------------------------------------------------

    for claim_type in claim_types:

        part = chunk[
            chunk["CLAIM_TYPE"] == claim_type
        ]

        if part.empty:
            continue

        row_counts[claim_type] += len(part)

        numeric_cols = part[
            numeric_candidates
        ].select_dtypes(
            include=[np.number]
        ).columns

        for col in numeric_cols:

            if col not in stats[claim_type]:

                stats[claim_type][col] = {
                    "values": [],
                    "zero": 0,
                    "negative": 0,
                    "missing": 0,
                    "min": np.inf,
                    "max": -np.inf
                }

            s = part[col]

            # ------------------------------------------------
            # Missing
            # ------------------------------------------------

            stats[claim_type][col]["missing"] += (
                s.isna().sum()
            )

            # ------------------------------------------------
            # Numeric non-missing values
            # ------------------------------------------------

            valid = s.dropna()

            if len(valid) == 0:
                continue

            # ------------------------------------------------
            # Store values temporarily for variance/cardinality
            # ------------------------------------------------

            stats[claim_type][col]["values"].append(
                valid
            )

            # ------------------------------------------------
            # Zero
            # ------------------------------------------------

            stats[claim_type][col]["zero"] += (
                (valid == 0).sum()
            )

            # ------------------------------------------------
            # Negative
            # ------------------------------------------------

            stats[claim_type][col]["negative"] += (
                (valid < 0).sum()
            )

            # ------------------------------------------------
            # Min / Max
            # ------------------------------------------------

            stats[claim_type][col]["min"] = min(
                stats[claim_type][col]["min"],
                valid.min()
            )

            stats[claim_type][col]["max"] = max(
                stats[claim_type][col]["max"],
                valid.max()
            )

# ============================================================
# CREATE FINAL AUDIT TABLES
# ============================================================

for claim_type in claim_types:

    print("\n")
    print("=" * 100)
    print(f"{claim_type} FEATURE AUDIT")
    print("=" * 100)

    print(
        "Rows:",
        f"{row_counts[claim_type]:,}"
    )

    records = []

    for col, info in stats[claim_type].items():

        if info["values"]:

            combined = pd.concat(
                info["values"],
                ignore_index=True
            )

            unique_count = combined.nunique()

            mean_value = combined.mean()

            std_value = combined.std()

            median_value = combined.median()

        else:

            unique_count = 0
            mean_value = np.nan
            std_value = np.nan
            median_value = np.nan

        records.append({
            "FEATURE": col,
            "UNIQUE": unique_count,
            "MEAN": mean_value,
            "STD": std_value,
            "MIN": (
                info["min"]
                if info["min"] != np.inf
                else np.nan
            ),
            "MEDIAN": median_value,
            "MAX": (
                info["max"]
                if info["max"] != -np.inf
                else np.nan
            ),
            "ZERO": info["zero"],
            "NEGATIVE": info["negative"],
            "MISSING": info["missing"]
        })

    audit = pd.DataFrame(records)

    # --------------------------------------------------------
    # Sort by unique values
    # --------------------------------------------------------

    print("\nCONSTANT FEATURES:")
    
    constant = audit[
        audit["UNIQUE"] <= 1
    ]

    if constant.empty:
        print("None")
    else:
        print(
            constant[
                ["FEATURE", "UNIQUE", "ZERO", "MISSING"]
            ].to_string(index=False)
        )

    # --------------------------------------------------------
    # Features with very low variation
    # --------------------------------------------------------

    print("\nLOW-CARDINALITY FEATURES:")

    low_card = audit[
        (audit["UNIQUE"] > 1) &
        (audit["UNIQUE"] <= 5)
    ].sort_values(
        "UNIQUE"
    )

    print(
        low_card[
            [
                "FEATURE",
                "UNIQUE",
                "MIN",
                "MEDIAN",
                "MAX"
            ]
        ].to_string(index=False)
    )

    # --------------------------------------------------------
    # Highest cardinality
    # --------------------------------------------------------

    print("\nHIGHEST-CARDINALITY FEATURES:")

    high_card = audit.sort_values(
        "UNIQUE",
        ascending=False
    ).head(20)

    print(
        high_card[
            [
                "FEATURE",
                "UNIQUE",
                "MIN",
                "MEDIAN",
                "MAX"
            ]
        ].to_string(index=False)
    )

    # --------------------------------------------------------
    # Save audit
    # --------------------------------------------------------

    output = BASE / (
        f"ml_feature_audit_{claim_type.lower()}.csv"
    )

    audit.to_csv(
        output,
        index=False
    )

    print(
        "\nSaved:",
        output
    )

print("\n")
print("=" * 100)
print("FULL CLAIM-TYPE-AWARE AUDIT COMPLETE")
print("=" * 100)

print("\nFinal row counts:")
print(row_counts)

Scanning chunk 1...
Scanning chunk 2...
Scanning chunk 3...
Scanning chunk 4...
Scanning chunk 5...
Scanning chunk 6...
Scanning chunk 7...
Scanning chunk 8...
Scanning chunk 9...
Scanning chunk 10...
Scanning chunk 11...
Scanning chunk 12...
Scanning chunk 13...
Scanning chunk 14...
Scanning chunk 15...
Scanning chunk 16...
Scanning chunk 17...
Scanning chunk 18...
Scanning chunk 19...
Scanning chunk 20...
Scanning chunk 21...
Scanning chunk 22...
Scanning chunk 23...


OUTPATIENT FEATURE AUDIT
Rows: 790,790

CONSTANT FEATURES:
                         FEATURE  UNIQUE   ZERO  MISSING
              AVG_CLAIM_DURATION       1 790790        0
             AVG_DIAGNOSIS_COUNT       1 790790        0
                     AVG_PAYMENT       1 790790        0
             AVG_PROCEDURE_COUNT       1 790790        0
          CLAIMS_PER_BENEFICIARY       1 790790        0
                     CLAIM_COUNT       1 790790        0
              CLAIM_DURATION_STD       1 790790        0
         

In [2]:
import pandas as pd
from pathlib import Path
import os

BASE = Path("../data/processed/primary")

print("Testing file access...")

files = [
    "outpatient_claims_with_beneficiary_features.csv",
    "inpatient_claims_fully_enriched.csv",
    "carrier_claims_with_beneficiary_features.csv",
]

for f in files:
    path = BASE / f

    size_gb = path.stat().st_size / (1024**3)

    print(f"{f}")
    print(f"  Size: {size_gb:.2f} GB")

Testing file access...
outpatient_claims_with_beneficiary_features.csv
  Size: 0.21 GB
inpatient_claims_fully_enriched.csv
  Size: 0.04 GB
carrier_claims_with_beneficiary_features.csv
  Size: 0.95 GB


In [4]:
import pandas as pd
from pathlib import Path

BASE = Path("../data/processed/primary")

INPUT = BASE / "outpatient_claims_with_beneficiary_features.csv"
OUTPUT = BASE / "outpatient_ml_ready.csv"

# ------------------------------------------------------------
# ONLY FEATURES NEEDED FOR OUTPATIENT MODEL
# ------------------------------------------------------------

COLS = [
    # IDs
    "CLAIM_KEY",
    "CLM_ID",
    "DESYNPUF_ID",

    # Time
    "CLAIM_YEAR",
    "CLAIM_MONTH",

    # Claim financial
    "CLM_PMT_AMT",
    "NCH_PRMRY_PYR_CLM_PD_AMT",
    "TOTAL_REIMBURSEMENT",
    "CLAIM_DURATION_DAYS",

    # Claim structure
    "DIAGNOSIS_COUNT",
    "PROCEDURE_COUNT",
    "HCPCS_COUNT",
    "HAS_DIAGNOSIS",
    "HAS_PROCEDURE",
    "HAS_HCPCS",
    "HAS_NEGATIVE_PAYMENT",
    "HAS_PRIMARY_PAYER_PAYMENT",
    "IS_SEGMENT_2",
    "HAS_SEGMENT_1_MATCH",

    # Beneficiary
    "BENE_SEX_IDENT_CD",
    "BENE_RACE_CD",
    "BENE_ESRD_IND",
    "SP_STATE_CODE",
    "BENE_COUNTY_CD",
    "BENE_HI_CVRAGE_TOT_MONS",
    "BENE_SMI_CVRAGE_TOT_MONS",
    "BENE_HMO_CVRAGE_TOT_MONS",
    "PLAN_CVRG_MOS_NUM",

    # Chronic conditions
    "SP_ALZHDMTA",
    "SP_CHF",
    "SP_CHRNKIDN",
    "SP_CNCR",
    "SP_COPD",
    "SP_DEPRESSN",
    "SP_DIABETES",
    "SP_ISCHMCHT",
    "SP_OSTEOPRS",
    "SP_RA_OA",
    "SP_STRKETIA",

    # Beneficiary reimbursement
    "MEDREIMB_IP",
    "BENRES_IP",
    "PPPYMT_IP",
    "MEDREIMB_OP",
    "BENRES_OP",
    "PPPYMT_OP",
    "MEDREIMB_CAR",
    "BENRES_CAR",
    "PPPYMT_CAR",
]

print("Reading outpatient...")

df = pd.read_csv(
    INPUT,
    usecols=COLS,
    low_memory=False
)

print("Loaded:", df.shape)

# ------------------------------------------------------------
# CONVERT ESRD
# ------------------------------------------------------------

df["BENE_ESRD_IND"] = (
    df["BENE_ESRD_IND"]
    .astype("string")
    .str.upper()
    .map({"0": 0, "Y": 1})
)

# ------------------------------------------------------------
# MISSINGNESS FLAGS
# ------------------------------------------------------------

for col in [
    "CLM_PMT_AMT",
    "TOTAL_REIMBURSEMENT",
    "CLAIM_DURATION_DAYS",
    "DIAGNOSIS_COUNT",
    "PROCEDURE_COUNT",
    "HCPCS_COUNT",
]:

    df[f"{col}_MISSING"] = (
        df[col].isna().astype("int8")
    )

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

df.to_csv(
    OUTPUT,
    index=False
)

print("=" * 80)
print("OUTPATIENT ML DATASET CREATED")
print("=" * 80)

print("Shape:", df.shape)
print("Saved:", OUTPUT)

del df

print("Outpatient memory released.")

Reading outpatient...
Loaded: (790790, 48)
OUTPATIENT ML DATASET CREATED
Shape: (790790, 54)
Saved: ..\data\processed\primary\outpatient_ml_ready.csv
Outpatient memory released.


In [5]:
import pandas as pd
from pathlib import Path

BASE = Path("../data/processed/primary")

INPUT = BASE / "inpatient_claims_fully_enriched.csv"
OUTPUT = BASE / "inpatient_ml_ready.csv"

COLS = [
    # IDs
    "CLAIM_KEY",
    "CLM_ID",
    "DESYNPUF_ID",

    # Time
    "CLAIM_YEAR",

    # Claim
    "CLM_PMT_AMT",
    "NCH_PRMRY_PYR_CLM_PD_AMT",
    "CLAIM_DURATION_DAYS",
    "DIAGNOSIS_COUNT",
    "PROCEDURE_COUNT",
    "HAS_NEGATIVE_PAYMENT",
    "HAS_PRIMARY_PAYER_PAYMENT",
    "CLM_UTLZTN_DAY_CNT",

    # Beneficiary
    "BENE_SEX_IDENT_CD",
    "BENE_RACE_CD",
    "BENE_ESRD_IND",
    "SP_STATE_CODE",
    "BENE_COUNTY_CD",
    "BENE_HI_CVRAGE_TOT_MONS",
    "BENE_SMI_CVRAGE_TOT_MONS",
    "BENE_HMO_CVRAGE_TOT_MONS",
    "PLAN_CVRG_MOS_NUM",

    # Chronic conditions
    "SP_ALZHDMTA",
    "SP_CHF",
    "SP_CHRNKIDN",
    "SP_CNCR",
    "SP_COPD",
    "SP_DEPRESSN",
    "SP_DIABETES",
    "SP_ISCHMCHT",
    "SP_OSTEOPRS",
    "SP_RA_OA",
    "SP_STRKETIA",

    # Beneficiary reimbursement
    "MEDREIMB_IP",
    "BENRES_IP",
    "PPPYMT_IP",
    "MEDREIMB_OP",
    "BENRES_OP",
    "PPPYMT_OP",
    "MEDREIMB_CAR",
    "BENRES_CAR",
    "PPPYMT_CAR",

    # Provider features
    "CLAIM_COUNT",
    "UNIQUE_CLAIM_COUNT",
    "TOTAL_PAYMENT",
    "AVG_PAYMENT",
    "MEDIAN_PAYMENT",
    "MAX_PAYMENT",
    "AVG_CLAIM_DURATION",
    "MAX_CLAIM_DURATION",
    "AVG_DIAGNOSIS_COUNT",
    "AVG_PROCEDURE_COUNT",
    "NEGATIVE_PAYMENT_COUNT",
    "PRIMARY_PAYER_PAYMENT_COUNT",
    "NEGATIVE_PAYMENT_RATE",
    "PRIMARY_PAYER_PAYMENT_RATE",
    "UNIQUE_BENEFICIARIES",
    "CLAIMS_PER_BENEFICIARY",
    "PAYMENT_STD",
    "CLAIM_DURATION_STD",
]

print("Reading inpatient...")

df = pd.read_csv(
    INPUT,
    usecols=COLS,
    low_memory=False
)

print("Loaded:", df.shape)

df["BENE_ESRD_IND"] = (
    df["BENE_ESRD_IND"]
    .astype("string")
    .str.upper()
    .map({"0": 0, "Y": 1})
)

for col in [
    "CLM_PMT_AMT",
    "CLAIM_DURATION_DAYS",
    "DIAGNOSIS_COUNT",
    "PROCEDURE_COUNT",
    "CLAIM_COUNT",
    "AVG_PAYMENT",
    "PAYMENT_STD",
]:

    df[f"{col}_MISSING"] = (
        df[col].isna().astype("int8")
    )

df.to_csv(
    OUTPUT,
    index=False
)

print("=" * 80)
print("INPATIENT ML DATASET CREATED")
print("=" * 80)

print("Shape:", df.shape)
print("Saved:", OUTPUT)

del df

print("Inpatient memory released.")

Reading inpatient...
Loaded: (66773, 59)
INPATIENT ML DATASET CREATED
Shape: (66773, 66)
Saved: ..\data\processed\primary\inpatient_ml_ready.csv
Inpatient memory released.


In [6]:
import pandas as pd
from pathlib import Path

BASE = Path("../data/processed/primary")

INPUT = BASE / "carrier_claims_with_beneficiary_features.csv"
OUTPUT = BASE / "carrier_ml_ready.csv"

COLS = [
    # IDs
    "CLM_ID",
    "DESYNPUF_ID",

    # Time
    "claim_year",
    "claim_month",
    "claim_day_of_week",

    # Carrier claim features
    "total_claim_payment_amt",
    "total_allowed_charge_amt",
    "total_deductible_amt",
    "total_coinsurance_amt",
    "total_primary_payer_paid_amt",
    "avg_payment_per_line",
    "payment_to_allowed_ratio",
    "line_count",
    "unique_hcpcs_count",
    "max_line_payment",
    "diagnosis_count",
    "unique_diagnosis_count",
    "distinct_provider_count_on_claim",
    "provider_claim_volume",
    "provider_avg_claim_payment",

    # Beneficiary
    "BENE_SEX_IDENT_CD",
    "BENE_RACE_CD",
    "BENE_ESRD_IND",
    "SP_STATE_CODE",
    "BENE_COUNTY_CD",
    "BENE_HI_CVRAGE_TOT_MONS",
    "BENE_SMI_CVRAGE_TOT_MONS",
    "BENE_HMO_CVRAGE_TOT_MONS",
    "PLAN_CVRG_MOS_NUM",

    # Chronic conditions
    "SP_ALZHDMTA",
    "SP_CHF",
    "SP_CHRNKIDN",
    "SP_CNCR",
    "SP_COPD",
    "SP_DEPRESSN",
    "SP_DIABETES",
    "SP_ISCHMCHT",
    "SP_OSTEOPRS",
    "SP_RA_OA",
    "SP_STRKETIA",

    # Beneficiary reimbursement
    "MEDREIMB_IP",
    "BENRES_IP",
    "PPPYMT_IP",
    "MEDREIMB_OP",
    "BENRES_OP",
    "PPPYMT_OP",
    "MEDREIMB_CAR",
    "BENRES_CAR",
    "PPPYMT_CAR",
]

print("Reading carrier...")
print("This is the largest file, so let it finish.")

df = pd.read_csv(
    INPUT,
    usecols=COLS,
    low_memory=False
)

print("Loaded:", df.shape)

df["BENE_ESRD_IND"] = (
    df["BENE_ESRD_IND"]
    .astype("string")
    .str.upper()
    .map({"0": 0, "Y": 1})
)

for col in [
    "total_claim_payment_amt",
    "total_allowed_charge_amt",
    "payment_to_allowed_ratio",
    "line_count",
    "diagnosis_count",
    "unique_hcpcs_count",
]:

    df[f"{col}_MISSING"] = (
        df[col].isna().astype("int8")
    )

df.to_csv(
    OUTPUT,
    index=False
)

print("=" * 80)
print("CARRIER ML DATASET CREATED")
print("=" * 80)

print("Shape:", df.shape)
print("Saved:", OUTPUT)

del df

print("Carrier memory released.")

Reading carrier...
This is the largest file, so let it finish.
Loaded: (4741335, 49)
CARRIER ML DATASET CREATED
Shape: (4741335, 55)
Saved: ..\data\processed\primary\carrier_ml_ready.csv
Carrier memory released.


In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../data/processed/primary")

files = {
    "OUTPATIENT": BASE / "outpatient_ml_ready.csv",
    "INPATIENT": BASE / "inpatient_ml_ready.csv",
    "CARRIER": BASE / "carrier_ml_ready.csv",
}

ID_COLS = [
    "CLAIM_KEY",
    "CLM_ID",
    "DESYNPUF_ID",
]

for name, file in files.items():

    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    # Read header first
    header = pd.read_csv(file, nrows=0)

    print("Columns:", len(header.columns))
    print("Column names:")
    print(list(header.columns))

    # Load only the dataset for this audit
    df = pd.read_csv(
        file,
        low_memory=False
    )

    print("\nShape:", df.shape)

    # --------------------------------------------------------
    # IDs
    # --------------------------------------------------------

    print("\nIDENTIFIERS")

    for col in ID_COLS:
        if col in df.columns:
            print(
                f"{col}: "
                f"unique={df[col].nunique(dropna=True):,} "
                f"missing={df[col].isna().sum():,}"
            )

    # --------------------------------------------------------
    # Missing
    # --------------------------------------------------------

    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    print("\nMISSING VALUES")

    if len(missing) == 0:
        print("None")
    else:
        print(missing)

    # --------------------------------------------------------
    # Infinite numeric values
    # --------------------------------------------------------

    numeric = df.select_dtypes(
        include=[np.number]
    )

    infinite_counts = np.isinf(numeric).sum()
    infinite_counts = infinite_counts[
        infinite_counts > 0
    ]

    print("\nINFINITE VALUES")

    if len(infinite_counts) == 0:
        print("None")
    else:
        print(infinite_counts)

    # --------------------------------------------------------
    # Constant features
    # --------------------------------------------------------

    constant = []

    for col in df.columns:

        if col in ID_COLS:
            continue

        if df[col].nunique(dropna=False) <= 1:
            constant.append(col)

    print("\nCONSTANT FEATURES")
    print(constant)

    # --------------------------------------------------------
    # Object / categorical columns
    # --------------------------------------------------------

    categorical = df.select_dtypes(
        include=["object", "string", "category"]
    ).columns.tolist()

    print("\nCATEGORICAL / OBJECT FEATURES")
    print(categorical)

    # --------------------------------------------------------
    # Numeric summary
    # --------------------------------------------------------

    print("\nNUMERIC FEATURE COUNT:", len(numeric.columns))

    print("\nTOP NUMERIC FEATURES BY SKEW")

    skew = numeric.skew(
        numeric_only=True
    ).sort_values(
        key=lambda x: x.abs(),
        ascending=False
    )

    print(skew.head(20))

    # --------------------------------------------------------
    # Release memory
    # --------------------------------------------------------

    del df

    print("\nMemory released for", name)


OUTPATIENT
Columns: 54
Column names:
['CLAIM_KEY', 'DESYNPUF_ID', 'CLM_ID', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT', 'TOTAL_REIMBURSEMENT', 'CLAIM_DURATION_DAYS', 'CLAIM_YEAR', 'CLAIM_MONTH', 'DIAGNOSIS_COUNT', 'PROCEDURE_COUNT', 'HCPCS_COUNT', 'HAS_DIAGNOSIS', 'HAS_PROCEDURE', 'HAS_HCPCS', 'HAS_NEGATIVE_PAYMENT', 'HAS_PRIMARY_PAYER_PAYMENT', 'IS_SEGMENT_2', 'HAS_SEGMENT_1_MATCH', 'BENE_SEX_IDENT_CD', 'BENE_RACE_CD', 'BENE_ESRD_IND', 'SP_STATE_CODE', 'BENE_COUNTY_CD', 'BENE_HI_CVRAGE_TOT_MONS', 'BENE_SMI_CVRAGE_TOT_MONS', 'BENE_HMO_CVRAGE_TOT_MONS', 'PLAN_CVRG_MOS_NUM', 'SP_ALZHDMTA', 'SP_CHF', 'SP_CHRNKIDN', 'SP_CNCR', 'SP_COPD', 'SP_DEPRESSN', 'SP_DIABETES', 'SP_ISCHMCHT', 'SP_OSTEOPRS', 'SP_RA_OA', 'SP_STRKETIA', 'MEDREIMB_IP', 'BENRES_IP', 'PPPYMT_IP', 'MEDREIMB_OP', 'BENRES_OP', 'PPPYMT_OP', 'MEDREIMB_CAR', 'BENRES_CAR', 'PPPYMT_CAR', 'CLM_PMT_AMT_MISSING', 'TOTAL_REIMBURSEMENT_MISSING', 'CLAIM_DURATION_DAYS_MISSING', 'DIAGNOSIS_COUNT_MISSING', 'PROCEDURE_COUNT_MISSING', 'HCPCS

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# PATHS
# ============================================================

BASE = Path("../data/processed/primary")

INPUT_FILE = BASE / "carrier_ml_ready.csv"
OUTPUT_FILE = BASE / "carrier_ml_preprocessed.csv"

# ============================================================
# FEATURES TO EXCLUDE
# ============================================================

ID_COLS = [
    "CLM_ID",
    "DESYNPUF_ID",
]

# These are constant according to the audit
CONSTANT_COLS = [
    "total_claim_payment_amt_MISSING",
    "total_allowed_charge_amt_MISSING",
    "payment_to_allowed_ratio_MISSING",
    "line_count_MISSING",
    "diagnosis_count_MISSING",
    "unique_hcpcs_count_MISSING",
]

# ============================================================
# COLUMNS WE WANT TO KEEP
# ============================================================

header = pd.read_csv(INPUT_FILE, nrows=0)
all_cols = list(header.columns)

DROP_COLS = [
    c for c in ID_COLS + CONSTANT_COLS
    if c in all_cols
]

FEATURE_COLS = [
    c for c in all_cols
    if c not in DROP_COLS
]

print("=" * 80)
print("CARRIER PREPROCESSING")
print("=" * 80)

print("Original columns:", len(all_cols))
print("Dropped columns:", len(DROP_COLS))
print("Final feature columns:", len(FEATURE_COLS))

print("\nDropped:")
for c in DROP_COLS:
    print(" ", c)

# ============================================================
# PROCESS IN CHUNKS
# ============================================================

CHUNK_SIZE = 250_000

if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()

first_chunk = True
total_rows = 0

# Read only the columns we actually need
for chunk_number, chunk in enumerate(
    pd.read_csv(
        INPUT_FILE,
        usecols=FEATURE_COLS,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    # --------------------------------------------------------
    # Convert ESRD indicator
    # --------------------------------------------------------

    if "BENE_ESRD_IND" in chunk.columns:
        chunk["BENE_ESRD_IND"] = (
            chunk["BENE_ESRD_IND"]
            .astype("string")
            .str.upper()
            .map({
                "0": 0,
                "Y": 1
            })
        )

    # --------------------------------------------------------
    # Convert object columns that should be numeric
    # --------------------------------------------------------

    for col in chunk.columns:

        if col in ["BENE_ESRD_IND"]:
            continue

        if chunk[col].dtype == "object":
            converted = pd.to_numeric(
                chunk[col],
                errors="coerce"
            )

            # Only replace if conversion actually worked
            if converted.notna().sum() > 0:
                chunk[col] = converted

    # --------------------------------------------------------
    # Replace infinite values
    # --------------------------------------------------------

    chunk = chunk.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # --------------------------------------------------------
    # Missing-value indicators for remaining missing values
    # --------------------------------------------------------

    important_missing_cols = [
        "BENE_SEX_IDENT_CD",
        "BENE_RACE_CD",
        "BENE_ESRD_IND",
        "SP_STATE_CODE",
        "BENE_COUNTY_CD",
        "PLAN_CVRG_MOS_NUM",
    ]

    for col in important_missing_cols:

        if col in chunk.columns:
            chunk[f"{col}_MISSING"] = (
                chunk[col].isna()
            ).astype("int8")

    # --------------------------------------------------------
    # Fill numeric missing values
    #
    # For Carrier, beneficiary matching was 100%, so any
    # remaining missing values are handled using medians.
    # --------------------------------------------------------

    numeric_cols = chunk.select_dtypes(
        include=[np.number]
    ).columns

    for col in numeric_cols:

        if chunk[col].isna().any():

            median_value = chunk[col].median()

            if pd.isna(median_value):
                median_value = 0

            chunk[col] = chunk[col].fillna(
                median_value
            )

    # --------------------------------------------------------
    # Save chunk
    # --------------------------------------------------------

    chunk.to_csv(
        OUTPUT_FILE,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False
    total_rows += len(chunk)

    print(
        f"Chunk {chunk_number}: "
        f"{total_rows:,} rows processed"
    )

print("\n" + "=" * 80)
print("CARRIER PREPROCESSING COMPLETE")
print("=" * 80)

print("Rows:", f"{total_rows:,}")
print("Saved:", OUTPUT_FILE)

CARRIER PREPROCESSING
Original columns: 55
Dropped columns: 8
Final feature columns: 47

Dropped:
  CLM_ID
  DESYNPUF_ID
  total_claim_payment_amt_MISSING
  total_allowed_charge_amt_MISSING
  payment_to_allowed_ratio_MISSING
  line_count_MISSING
  diagnosis_count_MISSING
  unique_hcpcs_count_MISSING
Chunk 1: 250,000 rows processed
Chunk 2: 500,000 rows processed
Chunk 3: 750,000 rows processed
Chunk 4: 1,000,000 rows processed
Chunk 5: 1,250,000 rows processed
Chunk 6: 1,500,000 rows processed
Chunk 7: 1,750,000 rows processed
Chunk 8: 2,000,000 rows processed
Chunk 9: 2,250,000 rows processed
Chunk 10: 2,500,000 rows processed
Chunk 11: 2,750,000 rows processed
Chunk 12: 3,000,000 rows processed
Chunk 13: 3,250,000 rows processed
Chunk 14: 3,500,000 rows processed
Chunk 15: 3,750,000 rows processed
Chunk 16: 4,000,000 rows processed
Chunk 17: 4,250,000 rows processed
Chunk 18: 4,500,000 rows processed
Chunk 19: 4,741,335 rows processed

CARRIER PREPROCESSING COMPLETE
Rows: 4,741,335
S

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../data/processed/primary")
FILE = BASE / "carrier_ml_preprocessed.csv"

print("=" * 80)
print("CARRIER PREPROCESSED DATASET VALIDATION")
print("=" * 80)

# ------------------------------------------------------------
# Read header
# ------------------------------------------------------------

header = pd.read_csv(FILE, nrows=0)

print("Columns:", len(header.columns))
print("\nColumns:")
print(list(header.columns))

# ------------------------------------------------------------
# Chunked validation
# ------------------------------------------------------------

CHUNK_SIZE = 250_000

total_rows = 0
missing_counts = None
infinite_counts = None

for i, chunk in enumerate(
    pd.read_csv(
        FILE,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    total_rows += len(chunk)

    # Missing
    current_missing = chunk.isna().sum()

    if missing_counts is None:
        missing_counts = current_missing
    else:
        missing_counts = missing_counts.add(
            current_missing,
            fill_value=0
        )

    # Infinite numeric values
    numeric = chunk.select_dtypes(
        include=[np.number]
    )

    current_inf = np.isinf(numeric).sum()

    if infinite_counts is None:
        infinite_counts = current_inf
    else:
        infinite_counts = infinite_counts.add(
            current_inf,
            fill_value=0
        )

    print(
        f"Validated {total_rows:,} rows..."
    )

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("VALIDATION RESULTS")
print("=" * 80)

print("Rows:", f"{total_rows:,}")
print("Expected:", f"{4_741_335:,}")
print("Row count correct:", total_rows == 4_741_335)

print("\nMissing values:")

missing_counts = missing_counts[
    missing_counts > 0
].sort_values(
    ascending=False
)

if len(missing_counts) == 0:
    print("None")
else:
    print(missing_counts)

print("\nInfinite values:")

infinite_counts = infinite_counts[
    infinite_counts > 0
].sort_values(
    ascending=False
)

if len(infinite_counts) == 0:
    print("None")
else:
    print(infinite_counts)

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)

CARRIER PREPROCESSED DATASET VALIDATION
Columns: 53

Columns:
['total_claim_payment_amt', 'total_allowed_charge_amt', 'total_deductible_amt', 'total_coinsurance_amt', 'total_primary_payer_paid_amt', 'avg_payment_per_line', 'payment_to_allowed_ratio', 'line_count', 'unique_hcpcs_count', 'max_line_payment', 'diagnosis_count', 'unique_diagnosis_count', 'distinct_provider_count_on_claim', 'provider_claim_volume', 'provider_avg_claim_payment', 'claim_year', 'claim_month', 'claim_day_of_week', 'BENE_SEX_IDENT_CD', 'BENE_RACE_CD', 'BENE_ESRD_IND', 'SP_STATE_CODE', 'BENE_COUNTY_CD', 'BENE_HI_CVRAGE_TOT_MONS', 'BENE_SMI_CVRAGE_TOT_MONS', 'BENE_HMO_CVRAGE_TOT_MONS', 'PLAN_CVRG_MOS_NUM', 'SP_ALZHDMTA', 'SP_CHF', 'SP_CHRNKIDN', 'SP_CNCR', 'SP_COPD', 'SP_DEPRESSN', 'SP_DIABETES', 'SP_ISCHMCHT', 'SP_OSTEOPRS', 'SP_RA_OA', 'SP_STRKETIA', 'MEDREIMB_IP', 'BENRES_IP', 'PPPYMT_IP', 'MEDREIMB_OP', 'BENRES_OP', 'PPPYMT_OP', 'MEDREIMB_CAR', 'BENRES_CAR', 'PPPYMT_CAR', 'BENE_SEX_IDENT_CD_MISSING', 'BENE_RACE

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../data/processed/primary")

INPUT_FILE = BASE / "outpatient_ml_ready.csv"
OUTPUT_FILE = BASE / "outpatient_ml_preprocessed.csv"

ID_COLS = [
    "CLAIM_KEY",
    "CLM_ID",
    "DESYNPUF_ID",
]

CONSTANT_COLS = [
    "CLM_PMT_AMT_MISSING",
    "TOTAL_REIMBURSEMENT_MISSING",
    "CLAIM_DURATION_DAYS_MISSING",
    "DIAGNOSIS_COUNT_MISSING",
    "PROCEDURE_COUNT_MISSING",
    "HCPCS_COUNT_MISSING",
]

header = pd.read_csv(INPUT_FILE, nrows=0)
all_cols = list(header.columns)

DROP_COLS = [
    c for c in ID_COLS + CONSTANT_COLS
    if c in all_cols
]

FEATURE_COLS = [
    c for c in all_cols
    if c not in DROP_COLS
]

print("=" * 80)
print("OUTPATIENT PREPROCESSING")
print("=" * 80)

print("Original columns:", len(all_cols))
print("Dropped columns:", len(DROP_COLS))
print("Final base features:", len(FEATURE_COLS))

if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()

CHUNK_SIZE = 250_000
first_chunk = True
total_rows = 0

for i, chunk in enumerate(
    pd.read_csv(
        INPUT_FILE,
        usecols=FEATURE_COLS,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    # Convert ESRD
    if "BENE_ESRD_IND" in chunk.columns:
        chunk["BENE_ESRD_IND"] = (
            chunk["BENE_ESRD_IND"]
            .astype("string")
            .str.upper()
            .map({"0": 0, "Y": 1})
        )

    # Convert object columns where possible
    for col in chunk.columns:
        if col == "BENE_ESRD_IND":
            continue

        if chunk[col].dtype == "object":
            converted = pd.to_numeric(
                chunk[col],
                errors="coerce"
            )

            if converted.notna().sum() > 0:
                chunk[col] = converted

    # Infinite → NaN
    chunk = chunk.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # Missingness indicators
    important_missing_cols = [
        "BENE_SEX_IDENT_CD",
        "BENE_RACE_CD",
        "BENE_ESRD_IND",
        "SP_STATE_CODE",
        "BENE_COUNTY_CD",
        "PLAN_CVRG_MOS_NUM",
    ]

    for col in important_missing_cols:
        if col in chunk.columns:
            chunk[f"{col}_MISSING"] = (
                chunk[col].isna()
            ).astype("int8")

    # Fill numeric missing values with chunk median
    numeric_cols = chunk.select_dtypes(
        include=[np.number]
    ).columns

    for col in numeric_cols:
        if chunk[col].isna().any():

            median_value = chunk[col].median()

            if pd.isna(median_value):
                median_value = 0

            chunk[col] = chunk[col].fillna(
                median_value
            )

    chunk.to_csv(
        OUTPUT_FILE,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False
    total_rows += len(chunk)

    print(f"Chunk {i}: {total_rows:,} rows processed")

print("\n" + "=" * 80)
print("OUTPATIENT PREPROCESSING COMPLETE")
print("=" * 80)

print("Rows:", f"{total_rows:,}")
print("Saved:", OUTPUT_FILE)

OUTPATIENT PREPROCESSING
Original columns: 54
Dropped columns: 9
Final base features: 45


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


Chunk 1: 250,000 rows processed


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


Chunk 2: 500,000 rows processed


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


Chunk 3: 750,000 rows processed


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


Chunk 4: 790,790 rows processed

OUTPATIENT PREPROCESSING COMPLETE
Rows: 790,790
Saved: ..\data\processed\primary\outpatient_ml_preprocessed.csv


In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../data/processed/primary")

INPUT_FILE = BASE / "inpatient_ml_ready.csv"
OUTPUT_FILE = BASE / "inpatient_ml_preprocessed.csv"

ID_COLS = [
    "CLAIM_KEY",
    "CLM_ID",
    "DESYNPUF_ID",
]

CONSTANT_COLS = [
    "CLM_PMT_AMT_MISSING",
    "CLAIM_DURATION_DAYS_MISSING",
    "DIAGNOSIS_COUNT_MISSING",
    "PROCEDURE_COUNT_MISSING",
    "CLAIM_COUNT_MISSING",
    "AVG_PAYMENT_MISSING",
    "PAYMENT_STD_MISSING",
]

header = pd.read_csv(INPUT_FILE, nrows=0)
all_cols = list(header.columns)

DROP_COLS = [
    c for c in ID_COLS + CONSTANT_COLS
    if c in all_cols
]

FEATURE_COLS = [
    c for c in all_cols
    if c not in DROP_COLS
]

print("=" * 80)
print("INPATIENT PREPROCESSING")
print("=" * 80)

print("Original columns:", len(all_cols))
print("Dropped columns:", len(DROP_COLS))
print("Final base features:", len(FEATURE_COLS))

if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()

CHUNK_SIZE = 100_000
first_chunk = True
total_rows = 0

for i, chunk in enumerate(
    pd.read_csv(
        INPUT_FILE,
        usecols=FEATURE_COLS,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    # Convert ESRD
    if "BENE_ESRD_IND" in chunk.columns:
        chunk["BENE_ESRD_IND"] = (
            chunk["BENE_ESRD_IND"]
            .astype("string")
            .str.upper()
            .map({"0": 0, "Y": 1})
        )

    # Convert object columns where possible
    for col in chunk.columns:
        if col == "BENE_ESRD_IND":
            continue

        if chunk[col].dtype == "object":
            converted = pd.to_numeric(
                chunk[col],
                errors="coerce"
            )

            if converted.notna().sum() > 0:
                chunk[col] = converted

    # Infinite → NaN
    chunk = chunk.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # Missingness indicators
    important_missing_cols = [
        "BENE_SEX_IDENT_CD",
        "BENE_RACE_CD",
        "BENE_ESRD_IND",
        "SP_STATE_CODE",
        "BENE_COUNTY_CD",
        "PLAN_CVRG_MOS_NUM",
    ]

    for col in important_missing_cols:
        if col in chunk.columns:
            chunk[f"{col}_MISSING"] = (
                chunk[col].isna()
            ).astype("int8")

    # Fill numeric missing values
    numeric_cols = chunk.select_dtypes(
        include=[np.number]
    ).columns

    for col in numeric_cols:
        if chunk[col].isna().any():

            median_value = chunk[col].median()

            if pd.isna(median_value):
                median_value = 0

            chunk[col] = chunk[col].fillna(
                median_value
            )

    chunk.to_csv(
        OUTPUT_FILE,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False
    total_rows += len(chunk)

    print(f"Chunk {i}: {total_rows:,} rows processed")

print("\n" + "=" * 80)
print("INPATIENT PREPROCESSING COMPLETE")
print("=" * 80)

print("Rows:", f"{total_rows:,}")
print("Saved:", OUTPUT_FILE)

INPATIENT PREPROCESSING
Original columns: 66
Dropped columns: 10
Final base features: 56


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


Chunk 1: 66,773 rows processed

INPATIENT PREPROCESSING COMPLETE
Rows: 66,773
Saved: ..\data\processed\primary\inpatient_ml_preprocessed.csv


In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../data/processed/primary")

FILES = {
    "CARRIER": BASE / "carrier_ml_preprocessed.csv",
    "OUTPATIENT": BASE / "outpatient_ml_preprocessed.csv",
    "INPATIENT": BASE / "inpatient_ml_preprocessed.csv",
}

EXPECTED_ROWS = {
    "CARRIER": 4_741_335,
    "OUTPATIENT": 790_790,
    "INPATIENT": 66_773,
}

for name, file in FILES.items():

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    header = pd.read_csv(file, nrows=0)

    print("Columns:", len(header.columns))

    total_rows = 0
    missing_counts = None
    infinite_counts = None

    for chunk in pd.read_csv(
        file,
        chunksize=250_000,
        low_memory=False
    ):

        total_rows += len(chunk)

        # Missing values
        current_missing = chunk.isna().sum()

        if missing_counts is None:
            missing_counts = current_missing
        else:
            missing_counts = missing_counts.add(
                current_missing,
                fill_value=0
            )

        # Infinite values
        numeric = chunk.select_dtypes(
            include=[np.number]
        )

        current_inf = np.isinf(numeric).sum()

        if infinite_counts is None:
            infinite_counts = current_inf
        else:
            infinite_counts = infinite_counts.add(
                current_inf,
                fill_value=0
            )

    missing_counts = missing_counts[
        missing_counts > 0
    ].sort_values(ascending=False)

    infinite_counts = infinite_counts[
        infinite_counts > 0
    ].sort_values(ascending=False)

    print("Rows:", f"{total_rows:,}")
    print("Expected:", f"{EXPECTED_ROWS[name]:,}")
    print(
        "Row count correct:",
        total_rows == EXPECTED_ROWS[name]
    )

    print("\nMissing values:")
    print(
        "None"
        if len(missing_counts) == 0
        else missing_counts
    )

    print("\nInfinite values:")
    print(
        "None"
        if len(infinite_counts) == 0
        else infinite_counts
    )

print("\n" + "=" * 80)
print("ALL ML DATASETS VALIDATION COMPLETE")
print("=" * 80)


CARRIER
Columns: 53
Rows: 4,741,335
Expected: 4,741,335
Row count correct: True

Missing values:
None

Infinite values:
None

OUTPATIENT
Columns: 51
Rows: 790,790
Expected: 790,790
Row count correct: True

Missing values:
None

Infinite values:
None

INPATIENT
Columns: 62
Rows: 66,773
Expected: 66,773
Row count correct: True

Missing values:
None

Infinite values:
None

ALL ML DATASETS VALIDATION COMPLETE
